<a href="https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Finding 1 — Freshness Multiplier:

The paper reports that freshness is associated with different growth-to-decline ratios across freshness windows, with 31–90 days showing the strongest stable ratio in the reported analysis. It also reports a separate 365+ page cohort where recently refreshed pages had higher health and impressions than older, less recently refreshed pages. My methodology question is: how was the refresh status defined, and does the comparison control for the fact that pages selected for refresh may already differ from untouched pages in visibility, quality, or historical performance?

Finding 2 — Age × Freshness Interaction:

The paper reports that content age and freshness interact, with a mature page that was recently updated appearing capable of competing with newer content. My methodology question is: are the age and freshness groups comparable enough that the observed health differences can be interpreted as an association rather than a selection effect, and how sensitive are the results to the exact bucket boundaries?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I first evaluate the Week-5 Random Forest with a row-level random split, which can place pages from the same client in both training and test sets. I then use a client-grouped holdout so entire clients are kept out of training when they are assigned to the test set. The grouped split is the more honest validation design for this lane because pages from one client can share traffic and content patterns. I compare the two setups using Precision@50 and average precision, while keeping the model features and evaluation definition fixed.

In [8]:
import os
import subprocess
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# =========================================================
# 1. LOAD STARTER DATA
# =========================================================

REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

# Clone the starter repository if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

data_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))


# =========================================================
# 2. CREATE THE OBSERVED TARGET
# =========================================================

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)


# =========================================================
# 3. DEFINE DECISION-TIME FEATURES
# =========================================================

feature_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

target_column = "is_declining_label"
group_column = "client_id"

model_data = df.dropna(
    subset=[target_column]
).copy()


# =========================================================
# 4. SAME RANDOM FOREST SPECIFICATION FOR BOTH SPLITS
# =========================================================

def build_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "random_forest",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ])


# =========================================================
# 5. PRECISION@50 FUNCTION
# =========================================================

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)

    top50 = y_true[order[:50]]

    return float(top50.mean())


# =========================================================
# 6. BEFORE — ROW-LEVEL RANDOM SPLIT
# =========================================================

X = model_data[feature_columns]
y = model_data[target_column]

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

random_model = build_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_precision50 = precision_at_50(
    y_test_random,
    random_scores
)

random_average_precision = average_precision_score(
    y_test_random,
    random_scores
)


# =========================================================
# 7. AFTER — CLIENT-GROUPED HOLDOUT
# =========================================================

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_splitter.split(
        model_data,
        model_data[target_column],
        groups=model_data[group_column]
    )
)

group_train = model_data.iloc[train_index].copy()
group_test = model_data.iloc[test_index].copy()

group_model = build_model()

group_model.fit(
    group_train[feature_columns],
    group_train[target_column]
)

group_scores = group_model.predict_proba(
    group_test[feature_columns]
)[:, 1]

group_precision50 = precision_at_50(
    group_test[target_column],
    group_scores
)

group_average_precision = average_precision_score(
    group_test[target_column],
    group_scores
)


# =========================================================
# 8. VERIFY NO CLIENT OVERLAP
# =========================================================

train_clients = set(
    group_train[group_column]
)

test_clients = set(
    group_test[group_column]
)

client_overlap = train_clients.intersection(
    test_clients
)

assert len(client_overlap) == 0


# =========================================================
# 9. BEFORE / AFTER COMPARISON
# =========================================================

before_after = pd.DataFrame({
    "validation_design": [
        "Before: row-level random split",
        "After: client-grouped holdout"
    ],
    "Precision@50": [
        random_precision50,
        group_precision50
    ],
    "Average Precision": [
        random_average_precision,
        group_average_precision
    ]
})

before_after["Precision@50"] = (
    before_after["Precision@50"]
    .round(3)
)

before_after["Average Precision"] = (
    before_after["Average Precision"]
    .round(3)
)


# =========================================================
# 10. PRINT RESULTS
# =========================================================

print("\nClient overlap in grouped split:", len(client_overlap))
print("Grouped split check: PASS")

print("\nBEFORE vs AFTER")
display(before_after)

print("\nRandom-split test rows:", len(X_test_random))
print("Grouped-split training clients:", len(train_clients))
print("Grouped-split test clients:", len(test_clients))

Rows: 30000
Columns: 44

Client overlap in grouped split: 0
Grouped split check: PASS

BEFORE vs AFTER


,validation_design,Precision@50,Average Precision
0,Before: row-level random split,0.88,0.733
1,After: client-grouped holdout,0.56,0.576



Random-split test rows: 6000
Grouped-split training clients: 25
Grouped-split test clients: 7


The row-level random split produced a Precision@50 of 0.880 and Average Precision of 0.733. After switching to a client-grouped holdout, Precision@50 fell to 0.560 and Average Precision to 0.576. The grouped split has zero client overlap between training and test sets, so it is the more conservative and credible estimate for this lane. The before/after gap suggests that the row-level split may benefit from shared client-specific patterns and therefore gives a more optimistic estimate of generalization.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# LEAKAGE AUDIT
# ---------------------------------------------------------

# Final Week-5 feature set
final_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

# Fields that would directly reveal the outcome
forbidden_fields = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_decline_answer",
    "product_flag",
    "needs_refresh",
    "needs_ctr_fix",
    "quick_win"
]

print("FINAL FEATURES")
for feature in final_features:
    print(" -", feature)

print("\nFORBIDDEN / LEAKAGE-PRONE FIELDS")
for field in forbidden_fields:
    print(" -", field)

# Check that no forbidden field is in the final feature set
leaked_features = set(final_features).intersection(
    forbidden_fields
)

print("\nForbidden fields used as features:", leaked_features)

assert len(leaked_features) == 0

print("Feature-list leakage check: PASS")

# ---------------------------------------------------------
# Check whether any final feature is obviously future-derived
# ---------------------------------------------------------

print("\nFuture-window / label-derived fields included: NO")
print("Final model features are defined from pre-decision page data only.")


FINAL FEATURES
 - impressions_90d
 - avg_position
 - ctr
 - content_age_days
 - days_since_last_update
 - word_count

FORBIDDEN / LEAKAGE-PRONE FIELDS
 - is_declining_label
 - trend_direction
 - trend_pct
 - future_decline_answer
 - product_flag
 - needs_refresh
 - needs_ctr_fix
 - quick_win

Forbidden fields used as features: set()
Feature-list leakage check: PASS

Future-window / label-derived fields included: NO
Final model features are defined from pre-decision page data only.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim:
The Random Forest is a better model for identifying declining pages than the Week-4 baseline.

Rewritten claim:
On the evaluated client-grouped holdout, the Random Forest did not outperform the Week-4 baseline: Precision@50 was 0.560 for the Random Forest versus 0.620 for the baseline. The earlier row-level random split produced a higher Precision@50 of 0.880, but that estimate is more optimistic because pages from the same clients can appear in both training and test sets. Therefore, the current evidence supports the Week-4 baseline as the stronger decision-support method on this evaluation setup, rather than showing that the Random Forest is generally better.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# CLAIM REWRITE — VERIFY THE REPORTED NUMBERS
# ---------------------------------------------------------

print("Claim-rewrite evidence")
print(
    "Random-split Precision@50:",
    round(random_precision50, 3)
)
print(
    "Client-grouped Precision@50:",
    round(group_precision50, 3)
)

print("\nWeek-4 baseline Precision@50: 0.620")
print("Week-5 Random Forest Precision@50: 0.560")

assert round(random_precision50, 3) == 0.880
assert round(group_precision50, 3) == 0.560

print("\nReported claim numbers verified.")


Claim-rewrite evidence
Random-split Precision@50: 0.88
Client-grouped Precision@50: 0.56

Week-4 baseline Precision@50: 0.620
Week-5 Random Forest Precision@50: 0.560

Reported claim numbers verified.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.